# 01 - Hello World: 基本設定與 LLM 呼叫 + MLflow Autolog

本筆記本示範如何載入設定、呼叫 LLM，並透過 MLflow autolog 自動追蹤。

## 本筆記本涵蓋內容

1. **載入設定**：使用 `init_config()` 從 `config/config.yaml` 載入專案設定
2. **初始化 MLflow**：啟用 autolog，自動追蹤 LLM 呼叫
3. **呼叫 LLM**：使用 `LLMClient` 連接 LLM 服務並發送請求

> **前置條件**：請確保已安裝相依套件（`uv sync`），並在 `config/config.yaml` 中填入正確的 API 設定。

In [ ]:
import sys
import os

# 將專案根目錄加入 Python 路徑
sys.path.insert(0, os.path.abspath(".."))

from app.utils.config import init_config
from app.tracking import init_mlflow, is_mlflow_available
from llm_service import LLMClient, LLMResponse

print("匯入完成。")

## 載入設定 & 初始化 MLflow

初始化後 MLflow 會自動啟用 LangChain autolog，所有 LLM 呼叫都會被追蹤。
也可以手動啟用特定框架的 autolog（如 `mlflow.openai.autolog()`）。

In [ ]:
# 初始化設定
cfg = init_config()

# 初始化 MLflow（啟用 autolog + set_active_model）
init_mlflow(cfg)

print(f"MLflow available: {is_mlflow_available()}")
print("\n=== 設定內容 ===")
for key, value in cfg.items():
    print(f"  {key}: {value}")

## 手動啟用其他框架的 Autolog（可選）

`init_mlflow()` 預設啟用 LangChain autolog。
如果直接使用 OpenAI SDK，可以額外啟用：

In [ ]:
import mlflow

# 如果使用 OpenAI SDK
# mlflow.openai.autolog()

# 如果使用 Anthropic SDK
# mlflow.anthropic.autolog()

# 全局啟用所有框架
# mlflow.autolog()

print("可依需求啟用其他框架的 autolog。")

## 呼叫 LLM

使用 `LLMClient.chat()` 發送對話請求。
由於已啟用 autolog，此呼叫會自動產生 trace 記錄。

> **注意**：需要一個正在運行的 LLM 服務（如 Ollama 或遠端 API）。

In [ ]:
# 建立 LLM Client
client = LLMClient(config={
    "base_url": cfg.model.base_url,
    "model": cfg.model.name,
    "temperature": cfg.model.temperature,
})

system_prompt = "你是一個有用的助手。"
user_prompt = "請用一句話介紹自己。"

try:
    response: LLMResponse = client.chat(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
    )

    print("=== LLM 回應 ===")
    print(f"內容     : {response.content}")
    print(f"模型     : {response.model}")
    print(f"Token 用量: {response.usage}")
    print(f"延遲     : {response.latency_ms:.1f} ms")

except Exception as e:
    print(f"[警告] LLM 呼叫失敗：{e}")
    print("請確認 LLM 服務已啟動。")

## 使用 Manual Span 追蹤自定義邏輯

除了 autolog 自動追蹤外，也可以使用 `mlflow.start_span()` 手動追蹤特定邏輯。

In [ ]:
import mlflow

@mlflow.trace
def my_pipeline(question: str) -> str:
    """帶有手動 span 的簡單 pipeline。"""
    with mlflow.start_span("preprocessing") as span:
        span.set_inputs({"raw_question": question})
        processed = question.strip().lower()
        span.set_outputs({"processed": processed})

    with mlflow.start_span("llm_call") as span:
        span.set_inputs({"prompt": processed})
        try:
            resp = client.chat("You are helpful.", processed)
            span.set_outputs({"response": resp.content})
            span.set_attributes({"model": resp.model, "tokens": resp.usage.total_tokens})
            return resp.content
        except Exception as e:
            span.set_attributes({"error": str(e)})
            return f"Error: {e}"

try:
    result = my_pipeline("  What is Python?  ")
    print(f"Result: {result}")
except Exception as e:
    print(f"[警告] Pipeline 執行失敗：{e}")

## 小結

| 步驟 | 函式 / 類別 | 說明 |
|------|------------|------|
| 載入設定 | `init_config()` | 載入 YAML 設定，支援環境變數解析 |
| 初始化 MLflow | `init_mlflow(cfg)` | 設定 tracking、啟用 autolog |
| 呼叫 LLM | `LLMClient.chat()` | 發送對話請求（自動追蹤） |
| 手動追蹤 | `@mlflow.trace` / `mlflow.start_span()` | 精細控制追蹤範圍 |

### 後續步驟

- `02_custom_workflow.ipynb`：使用 LangGraph 建構多步驟 workflow
- `03_evaluation.ipynb`：使用 MLflow GenAI evaluate 進行評估
- `04_prompt_management.ipynb`：Prompt 註冊、版本管理與自動優化
- 啟動 MLflow UI 查看 traces：`mlflow ui --port 5000`